# 🤖 Agentic Research Assistant

An autonomous, tool-using AI agent that uses a **Planner Agent** (powered by an LLM) to decide *which tools to call, in what order, and whether to call them at all* based on what the user actually asks for -- instead of running a fixed pipeline every time.


## Architecture

```
User Query
    |
    v
Planner Agent (Groq LLM -- reasons about intent, decides which tools to call)
    |
    +--> search_papers        (FAISS semantic retrieval)
    +--> summarize_paper      (BART abstractive summary)
    +--> extract_keywords     (KeyBERT)
    +--> extract_entities     (NER)
    +--> compare_papers       (compares two papers side by side)
    +--> generate_report      (exports a PDF report)
    |
    v
Final Response (text answer, or a generated PDF -- whatever the query actually needs)
```

**Why this is agentic and not just a pipeline:** the old project always ran search -> summarize -> keywords -> entities -> PDF in a fixed sequence for every query. Here, a *single* query like "just summarize this paper" only triggers `search_papers` and `summarize_paper` -- the agent skips the rest because it reasoned that's all the user needs. A query like "compare these two papers" routes to a completely different tool. The agent also keeps conversation memory, so a follow-up like "now compare it with the second one" still works.


## 1. Setup & Installation

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu transformers==4.46.3 \
    keybert==0.8.5 spacy scikit-learn reportlab rich \
    langchain langchain-core langchain-community langchain-groq
!python -m spacy download en_core_web_sm -q


## 2. API Key

This project uses [Groq](https://console.groq.com/) as the LLM provider for the Planner Agent -- it's free to sign up and fast for tool-calling models like Llama 3.1.

- **In Google Colab**: add `GROQ_API_KEY` under the key-shaped Secrets tab (left sidebar), then run the cell below.
- **Locally**: set it as an environment variable before launching Jupyter, e.g. `export GROQ_API_KEY=your_key` (PowerShell: `$env:GROQ_API_KEY="your_key"`), or paste it directly (not recommended for shared notebooks).


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Loaded GROQ_API_KEY from Colab secrets")
except ImportError:
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = input("Paste your Groq API key: ").strip()
    print("Using GROQ_API_KEY from environment")


## 3. Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

from transformers import pipeline
from keybert import KeyBERT
import spacy

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table as PDFTable, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch

from rich.console import Console
from rich.panel import Panel

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_groq import ChatGroq

console = Console()


## 4. Load Data, Embeddings & FAISS Index

This block loads the dataset, embeddings, and FAISS index using a caching strategy, so re-running the notebook doesn't recompute everything from scratch.


In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
df = pd.DataFrame(dataset)[["title", "abstract"]].head(15000).reset_index(drop=True)
df["paper_text"] = (df["title"] + " " + df["abstract"]).str.replace("\n", " ", regex=False).str.strip()
print(df.shape)


In [ ]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists("paper_embeddings.npy"):
    print("Loading cached embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = embed_model.encode(df["paper_text"].tolist(), show_progress_bar=True, convert_to_numpy=True)
    np.save("paper_embeddings.npy", embeddings)

if os.path.exists("paper_faiss.index"):
    print("Loading cached FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Building FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")

print("Vectors indexed:", index.ntotal)


## 5. Load Supporting Models

Summarizer, keyword extractor, and NER models used by the agent's tools.

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
kw_model = KeyBERT(embed_model)
ner_model = spacy.load("en_core_web_sm")
print("Models loaded")


## 6. Core Helper Functions

These are plain Python functions -- the actual capabilities. In the next section we wrap each one as a LangChain `@tool` so the agent can call them.


In [ ]:
def raw_search(query, k=5):
    """Return top-k (score, index) pairs for a query."""
    query_embedding = embed_model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    return list(zip(D[0].tolist(), I[0].tolist()))


def raw_summarize(text, max_length=120, min_length=40):
    result = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
    return result[0]["summary_text"]


def raw_keywords(text, top_n=5):
    kws = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words="english", top_n=top_n)
    return [k for k, _ in kws]


def raw_entities(text):
    doc = ner_model(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


## 7. Wrap Capabilities as LangChain Tools

Each tool has a clear docstring -- this is what the Planner Agent reads to decide when to use it. Vague docstrings lead to a confused agent, so be specific about what each tool does and doesn't do.


In [ ]:
@tool
def search_papers(query: str, k: int = 3) -> str:
    """
    Search the research paper database for papers relevant to a topic or question.
    Returns the title, similarity score, and a short excerpt of the abstract for each
    of the top-k matches. Use this whenever the user wants to find papers on a topic.
    """
    results = raw_search(query, k)
    lines = []
    for score, idx in results:
        row = df.iloc[idx]
        lines.append(
            f"[paper_id={idx}] score={score:.3f} | title=\"{row['title'].strip()}\" | "
            f"abstract_excerpt=\"{row['abstract'][:200].strip()}...\""
        )
    return "\n".join(lines) if lines else "No matching papers found."


In [ ]:
@tool
def summarize_paper(paper_id: int) -> str:
    """
    Generate a concise AI summary of a paper's abstract, given its paper_id
    (obtained from search_papers). Use this when the user wants a summary
    of a specific paper rather than the full abstract.
    """
    if paper_id < 0 or paper_id >= len(df):
        return f"Invalid paper_id: {paper_id}"
    abstract = df.iloc[paper_id]["abstract"]
    return raw_summarize(abstract)


In [ ]:
@tool
def extract_keywords(paper_id: int, top_n: int = 5) -> str:
    """
    Extract the top_n most relevant keywords/keyphrases from a paper's abstract,
    given its paper_id. Use this when the user asks for keywords, topics, or
    key terms of a paper.
    """
    if paper_id < 0 or paper_id >= len(df):
        return f"Invalid paper_id: {paper_id}"
    abstract = df.iloc[paper_id]["abstract"]
    keywords = raw_keywords(abstract, top_n)
    return ", ".join(keywords)


In [ ]:
@tool
def extract_entities(paper_id: int) -> str:
    """
    Extract named entities (organizations, technologies, locations, etc.) from a
    paper's abstract, given its paper_id. Use this when the user asks what
    organizations, models, or entities are mentioned in a paper.
    """
    if paper_id < 0 or paper_id >= len(df):
        return f"Invalid paper_id: {paper_id}"
    abstract = df.iloc[paper_id]["abstract"]
    entities = raw_entities(abstract)
    if not entities:
        return "No named entities found."
    return ", ".join(f"{text} ({label})" for text, label in entities)


In [ ]:
@tool
def compare_papers(paper_id_1: int, paper_id_2: int) -> str:
    """
    Compare two papers side by side, given their paper_ids (obtained from
    search_papers). Returns each paper's title, a short summary, and their
    cosine similarity to each other. Use this when the user wants to compare,
    contrast, or see how related two specific papers are.
    """
    for pid in (paper_id_1, paper_id_2):
        if pid < 0 or pid >= len(df):
            return f"Invalid paper_id: {pid}"

    row1, row2 = df.iloc[paper_id_1], df.iloc[paper_id_2]
    emb1 = embed_model.encode([row1["paper_text"]])
    emb2 = embed_model.encode([row2["paper_text"]])
    faiss.normalize_L2(emb1)
    faiss.normalize_L2(emb2)
    similarity = float(np.dot(emb1[0], emb2[0]))

    summary1 = raw_summarize(row1["abstract"])
    summary2 = raw_summarize(row2["abstract"])

    return (
        f"Paper A: \"{row1['title'].strip()}\"\nSummary A: {summary1}\n\n"
        f"Paper B: \"{row2['title'].strip()}\"\nSummary B: {summary2}\n\n"
        f"Cosine similarity between the two papers: {similarity:.3f}"
    )


In [ ]:
@tool
def generate_report(paper_id: int, query: str) -> str:
    """
    Generate a downloadable PDF report for a paper (title, summary, keywords,
    entities) given its paper_id and the original user query. Use this only
    when the user explicitly asks for a report, PDF, or document to download.
    """
    if paper_id < 0 or paper_id >= len(df):
        return f"Invalid paper_id: {paper_id}"

    row = df.iloc[paper_id]
    summary = raw_summarize(row["abstract"])
    keywords = raw_keywords(row["abstract"])
    entities = raw_entities(row["abstract"])

    filename = f"Agent_Report_{query[:30].strip().replace(' ', '_')}.pdf"
    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(filename)
    elements = [
        Paragraph("Agentic Research Assistant -- Report", styles["Title"]),
        Spacer(1, 12),
        Paragraph(f"<b>Query:</b> {query}", styles["Normal"]),
        Paragraph(f"<b>Paper:</b> {row['title'].strip()}", styles["Normal"]),
        Spacer(1, 12),
        Paragraph(f"<b>Summary:</b> {summary}", styles["Normal"]),
        Spacer(1, 12),
        Paragraph(f"<b>Keywords:</b> {', '.join(keywords)}", styles["Normal"]),
        Spacer(1, 12),
        Paragraph(
            f"<b>Entities:</b> {', '.join(f'{t} ({l})' for t, l in entities) or 'None found'}",
            styles["Normal"],
        ),
    ]
    doc.build(elements)
    return f"Report generated: {filename}"


In [ ]:
tools = [
    search_papers,
    summarize_paper,
    extract_keywords,
    extract_entities,
    compare_papers,
    generate_report,
]
print(f"{len(tools)} tools registered:", [t.name for t in tools])


## 8. Planner Agent

The LLM is given a system prompt establishing it as a Planner Agent, then bound to the tool set. This is the piece that decides *what* to call, *when*, and *how many times* -- not a hardcoded sequence.


In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
llm_with_tools = llm.bind_tools(tools)


In [ ]:
SYSTEM_PROMPT = SystemMessage(content="""
You are a Planner Agent for a research paper assistant.

You have access to tools that can search papers, summarize them, extract keywords,
extract entities, compare two papers, and generate a PDF report.

Rules:
1. Only call the tools that are actually needed to answer the user's request.
   Do NOT run every tool for every query.
2. To summarize, extract keywords/entities from, or generate a report for a paper,
   you first need its paper_id -- get this by calling search_papers first if you
   don't already have it from earlier in the conversation.
3. When comparing papers, make sure you have both paper_ids before calling compare_papers.
4. Never invent a paper_id, title, or numeric result -- only use what tools return.
5. After getting tool results, answer the user directly and clearly using that data.
6. Only call generate_report if the user explicitly asks for a report, PDF, or document.
""")


## 9. Agent Loop (with Conversation Memory)

This is a manual agentic loop: the LLM proposes tool calls, we execute them, feed the results back, and let the LLM decide whether it needs to call more tools or is ready to answer. Conversation history persists across calls, so follow-up questions work (e.g. "now compare that with the second result").


In [ ]:
conversation_history = [SYSTEM_PROMPT]


def run_agent(user_input, max_tool_iterations=5, verbose=True):
    """Send a user message through the agent loop, executing any tool calls
    the LLM requests, and return the final text answer."""
    conversation_history.append(HumanMessage(content=user_input))

    for _ in range(max_tool_iterations):
        response = llm_with_tools.invoke(conversation_history)
        conversation_history.append(response)

        if not response.tool_calls:
            if verbose:
                console.print(Panel(response.content, title="Agent Answer", border_style="green"))
            return response.content

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            matching_tool = next((t for t in tools if t.name == tool_name), None)

            if verbose:
                console.print(f"[yellow]-> Agent is calling:[/yellow] {tool_name}({tool_args})")

            if matching_tool is None:
                tool_result = f"Error: no such tool '{tool_name}'"
            else:
                try:
                    tool_result = matching_tool.invoke(tool_args)
                except Exception as e:
                    tool_result = f"Tool error: {e}"

            conversation_history.append(
                ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
            )

    return "Reached max tool iterations without a final answer."


## 10. Demo

Notice how different queries trigger different tool combinations -- that's the agentic behavior in action.


In [ ]:
run_agent("Find me 3 papers about vision transformers")


In [ ]:
run_agent("Summarize the first one")


In [ ]:
run_agent("What are the keywords and entities in that paper?")


In [ ]:
run_agent("Compare the first and second papers you found earlier")


In [ ]:
run_agent("Generate a PDF report for the first paper")


## 11. Reset Conversation

Run this to clear memory and start a fresh conversation.


In [ ]:
conversation_history = [SYSTEM_PROMPT]
print("Conversation memory cleared.")
